# DVFopt — unified deformation-field optimization

End-to-end demo of the unified `DVFopt` class:

- Configure constraint (`2tri` / `jdet`), solver (`slsqp` / `trust-constr` / `barrier` / `auto`), objective (`l2` / `l1` / `none`), decomposition (`windowed` / `full-grid`) and the various add-ons (continuation, perturb-on-stall, L1 polish, etc.) in one `DVFoptConfig`.
- One `fit(deformation)` call → corrected DVF + a `Result` with per-slice / per-iteration history and a tabular summary.
- Built-in plots: convergence curves, feasibility heatmap + histogram (the "feasibility wall"), and constraint-gradient stiffness map (where SLSQP's active-set line search degenerates).

Input data is the registration field at `data/corrected_correspondences_count_touching/registered_output/deformation3d.npy`.

In [ ]:
import os, sys, time
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from dvfopt import DVFopt, DVFoptConfig

DATA_PATH = os.path.abspath(os.path.join(
    '..', 'data', 'corrected_correspondences_count_touching',
    'registered_output', 'deformation3d.npy'))
phi = np.load(DATA_PATH)
print('phi.shape =', phi.shape)

## Example 1 — minimal: just fit a slice

Defaults: `constraint='2tri'`, `solver='auto'`, `objective='l2'`, `threshold=0.01`, `mode='windowed'`.

In [ ]:
opt = DVFopt(constraint='2tri', solver='slsqp', threshold=0.01,
             record_history=True, record_snapshots=True, verbose=1)
# z=126 is a moderately-folded slice (178 init folds).
result = opt.fit(phi[:, 126:127])
print(result.summary())
print()
print(result.to_dataframe())

## Example 2 — switch solver/objective/threshold per fit

The barrier solver (`solver='barrier'`) drives the constraint as one coupled full-grid problem — no per-component windowing. With `objective='l2'` and `threshold=0` it can be used to push a heavily-folded slice as close to feasible as the geometry allows (see the experiments notebooks for the full analysis).

In [ ]:
# Same slice, barrier solver instead, looser threshold.
opt_b = DVFopt(constraint='2tri', solver='barrier', objective='l2',
               threshold=0.0, margin=1e-3, mode='full-grid',
               record_history=True, record_snapshots=True, verbose=1,
               # Cap the schedule short for the demo; bump for real runs.
               lam_schedule=(1.0, 10.0, 100.0, 1e3, 1e4, 1e5, 1e6),
               barrier_max_iter=200)
result_b = opt_b.fit(phi[:, 126:127])
print(result_b.summary())

## Example 3 — toggle objective (L2 vs L1)

Smoothed L1 anchor promotes sparse corrections (few large moves) vs L2's spread (many small moves). On most slices L1 and L2 give similar end states; on the geometrically-stuck dense slices the experiments showed they hit the same wall (see `slsqp_degeneracy_at_zero.ipynb`).

In [ ]:
opt_l1 = DVFopt(constraint='2tri', solver='barrier', objective='l1',
                threshold=0.01, mode='full-grid',
                record_history=True, verbose=1,
                lam_schedule=(1.0, 10.0, 100.0, 1e3, 1e4, 1e5, 1e6),
                barrier_max_iter=200)
result_l1 = opt_l1.fit(phi[:, 126:127])
print(result_l1.summary())

## Visualizations

Three built-in plots on any `Result`:

- `plot_convergence(z)` — fold count and worst-constraint value vs iteration.
- `plot_feasibility(z)` — heatmap of the constraint field + histogram with the feasibility threshold marked ("the wall").
- `plot_gradient_region(z)` — per-cell `|T| / ||grad T||`, i.e. the Newton-step magnitude required to move T by O(T). Bright regions = stiff / near-degenerate = where SLSQP's active-set line search collapses.

In [ ]:
# Note: when we passed phi[:, 126:127] (a single-slice view), the Result
# indexes its slices 0..N-1 by *internal* position, not absolute volume z.
# So this single-slice Result uses z=0.
result_b.plot_convergence(z=0)

In [ ]:
result_b.plot_feasibility(z=0, snapshot=0)    # init snapshot
result_b.plot_feasibility(z=0, snapshot=-1)   # final snapshot

In [ ]:
# Per-cell constraint-gradient stiffness on the *corrected* slice.
# Highlighted regions are where the 2-triangle Jacobian is near-rank-
# deficient -- the area where SLSQP plateaus.
result_b.plot_gradient_region(z=0)

## Tabular outputs

- `result.to_dataframe()` — one row per slice with init/final stats.
- `result.history_df()` — long-form, one row per iteration (when `record_history=True`).

In [ ]:
result_b.to_dataframe()

In [ ]:
result_b.history_df().head(15)

## All available configuration

Inspect `DVFoptConfig` for the full set of knobs (constraint, solver, objective, mode, decomposition pads, outer iter cap, SLSQP knobs, barrier schedules, jacobian mode, verbosity, history recording).

Any field can be overridden either by constructing a `DVFoptConfig` and passing it in, or by keyword arguments to `DVFopt(...)`.

In [ ]:
import dataclasses
for f in dataclasses.fields(DVFoptConfig):
    print(f'{f.name:25s}  default = {f.default}')